In [0]:
# Databricks notebook source
# =============================================================
#  GEO SAFRAS · 02_pragas_silver_transform.py
#  Transformação Bronze → Silver
#  • Todos os campos STRING em minúsculo
#  • Tipagem numérica com DECIMAL
#  • Deduplicação com MERGE
#  • Colunas de auditoria (updated_at)
# =============================================================

# COMMAND ----------
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from datetime import datetime

CATALOG  = "workspace"
SCHEMA_B = "gs_bronze"
SCHEMA_S = "gs_silver"
NOW      = F.lit(datetime.now().isoformat()).cast("timestamp")

print(f"[Silver Pragas] Início: {datetime.now().isoformat()}")

# COMMAND ----------
df_b = spark.table(f"{CATALOG}.{SCHEMA_B}.pragas_defensivos_culturas")

df_s = (df_b
    # Campos STRING em minúsculo
    .withColumn("cultura",                    F.lower(F.trim(F.col("cultura"))))
    .withColumn("praga",                      F.lower(F.trim(F.col("praga"))))
    .withColumn("grupo_praga",                F.lower(F.trim(F.col("grupo_praga"))))
    .withColumn("nivel_infestacao",           F.lower(F.trim(F.col("nivel_infestacao"))))
    .withColumn("principio_ativo_recomendado",F.lower(F.trim(F.col("principio_ativo_recomendado"))))
    .withColumn("intervalo_aplicacao_dias",   F.lower(F.trim(F.col("intervalo_aplicacao_dias"))))
    .withColumn("observacao",                 F.lower(F.trim(F.col("observacao"))))
    .withColumn("source_file",                F.lower(F.trim(F.col("source_file"))))

    # Tipagem numérica
    .withColumn("intensidade_min",       F.col("intensidade_min").cast("decimal(6,2)"))
    .withColumn("intensidade_max",       F.col("intensidade_max").cast("decimal(6,2)"))
    .withColumn("dose_inseticida_L_ha",  F.col("dose_inseticida_L_ha").cast("decimal(8,3)"))
    .withColumn("dose_inseticida_L_100ha", F.col("dose_inseticida_L_100ha").cast("decimal(10,2)"))
    .withColumn("dose_inseticida_L_500ha", F.col("dose_inseticida_L_500ha").cast("decimal(10,2)"))
    .withColumn("dose_inseticida_L_1000ha",F.col("dose_inseticida_L_1000ha").cast("decimal(10,2)"))

    # Auditoria
    .withColumn("updated_at", NOW)
    .drop("ingestion_timestamp")
)

# Validação — não deve ter nulos em campos chave
invalidos = df_s.filter(
    F.col("cultura").isNull() |
    F.col("praga").isNull() |
    F.col("nivel_infestacao").isNull()
).count()
if invalidos > 0:
    print(f"  ⚠️  {invalidos} registros com campos chave nulos — ignorados")
df_s = df_s.filter(
    F.col("cultura").isNotNull() &
    F.col("praga").isNotNull() &
    F.col("nivel_infestacao").isNotNull()
)

print(f"  ✓ pragas Silver: {df_s.count()} registros prontos")

# COMMAND ----------
# MERGE Silver — chave: cultura + praga + nivel_infestacao
cols = {c: f"src.{c}" for c in df_s.columns}
chave = """
    tgt.cultura          = src.cultura          AND
    tgt.praga            = src.praga            AND
    tgt.nivel_infestacao = src.nivel_infestacao
"""

if spark.catalog.tableExists(f"{CATALOG}.{SCHEMA_S}.pragas_defensivos_culturas"):
    dt = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA_S}.pragas_defensivos_culturas")
    (dt.alias("tgt")
       .merge(df_s.alias("src"), chave)
       .whenMatchedUpdate(set=cols)
       .whenNotMatchedInsert(values=cols)
       .execute())
    print("  ✓ pragas_defensivos_culturas Silver: MERGE concluído")
else:
    df_s.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA_S}.pragas_defensivos_culturas")
    print("  ✓ pragas_defensivos_culturas Silver: criada e carregada")

# COMMAND ----------
print(f"\n[Silver Pragas] ✅ Concluído em {datetime.now().isoformat()}")
print(f"  • {CATALOG}.{SCHEMA_S}.pragas_defensivos_culturas")